In [3]:
import stim
import numpy as np
import matplotlib.pyplot as plt
import pymatching
import sys
import os
import CircuitPresets as cp
import CircuitBuilder
import random
import pickle

from tqdm import tqdm

In [6]:
# Original [[15, 7, 3]] quantum hamming code with 8 stabilizers
# Stabilizer measurement schedule requires 8 time windows:
# 12345678
# 4123    5678
# 56  12  78  34
# 6 5 2 1 3 4 7 8
# 34127856
# 2341    6785
# 78  34  12  56
# 8 7 4 3 2 1 6 5


def hamming15(noise, nrounds):
    cb = CircuitBuilder.CircuitBuilder(noise, noise, noise, noise, noise, noise)

    # State preparation
    cb.al('R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    cb.al('R 15 16 17 18 19 20 21 22')
    # Initial stabilizer checks, just measure the 8 stabilizers which should all be 0 initially
    cb.al('H 19 20 21 22')


    cb.begin_loop(nrounds)
    # Time window 1
    cb.time_tick()
    cb.ale('CX 0 15 1 16 4 17 6 18') # Z
    cb.ale('CX 19 2 20 3 21 8 22 10') # X
    # Time window 2
    cb.time_tick()
    cb.ale('CX 1 15 2 16 5 17 4 18')
    cb.ale('CX 19 3 20 0 21 9 22 8')
    # Time window 3
    cb.time_tick()
    cb.ale('CX 2 15 3 16 12 17 8 18')
    cb.ale('CX 19 0 20 1 21 4 22 6')
    # Time window 4
    cb.time_tick()
    cb.ale('CX 3 15 0 16 13 17 10 18')
    cb.ale('CX 19 1 20 2 21 5 22 4')
    # Time window 5
    cb.time_tick()
    cb.ale('CX 4 15 8 16 0 17 2 18')
    cb.ale('CX 19 6 20 11 21 12 22 14')
    # Time window 6
    cb.time_tick()
    cb.ale('CX 5 15 9 16 1 17 0 18')
    cb.ale('CX 19 7 20 8 21 13 22 12')
    # Time window 7
    cb.time_tick()
    cb.ale('CX 6 15 10 16 8 17 12 18')
    cb.ale('CX 19 4 20 9 21 0 22 2')
    # Time window 8
    cb.time_tick()
    cb.ale('CX 7 15 11 16 9 17 14 18')
    cb.ale('CX 19 5 20 10 21 1 22 0')

    # Measurement
    cb.ale('H 19 20 21 22')
    cb.ale(f'MR({noise}) 15 16 17 18 19 20 21 22')
    cb.ale('H 19 20 21 22')
    cb.add_detectors([[-1, -9], [-2, -10], [-3, -11], [-4, -12]]) # X type stabilizers
    cb.add_detectors([[-5, -13], [-6, -14], [-7, -15], [-8, -16]]) # Z type stabilizers

    cb.end_loop()

    # Final measurement of all physical qubits to give logical readout
    #cb.ale(f'MR(0) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    #cb.add_detector([-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15])
    return cb


In [7]:
# Second attempt at a quantum hamming code that works:

def hamming15(noise, nrounds):
    cb = CircuitBuilder.CircuitBuilder(noise, noise, noise, noise, noise, noise)

    # State preparation
    cb.al('R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    #cb.al('X 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14') # For preparing all 1s
    cb.al('R 15 16 17 18 19 20 21 22')
    cb.al('H 19 20 21 22') # Rotate X type ancilla to X basis
    # Initial stabilizer checks
    # Z stabilizers use ancilla 15 16 17 18
    # 12345678
    # 2345    6781
    # 34  67  12  58
    # 4 5 1 2 3 6 7 8
    cb.time_tick()
    cb.ale('CX 0 15 11 16 8 17 4 18')
    cb.time_tick()
    cb.ale('CX 1 15 0 16 9 17 6 18')
    cb.time_tick()
    cb.ale('CX 2 15 1 16 0 17 8 18')
    cb.time_tick()
    cb.ale('CX 3 15 2 16 1 17 0 18')
    cb.time_tick()
    cb.ale('CX 4 15 3 16 12 17 2 18')
    cb.time_tick()
    cb.ale('CX 5 15 8 16 4 17 10 18')
    cb.time_tick()
    cb.ale('CX 6 15 9 16 5 17 12 18')
    cb.time_tick()
    cb.ale('CX 7 15 10 16 13 17 14 18')
    #cb.ale(f'MR({noise}) 15 16 17 18')
    cb.al('MR(0) 15 16 17 18')   
    # X stabilizers use ancilla 19 20 21 22 in X basis, same schedule
    cb.time_tick()
    cb.ale('CX 19 0 20 11 21 8 22 4')
    cb.time_tick()
    cb.ale('CX 19 1 20 0 21 9 22 6')
    cb.time_tick()
    cb.ale('CX 19 2 20 1 21 0 22 8')
    cb.time_tick()
    cb.ale('CX 19 3 20 2 21 1 22 0')
    cb.time_tick()
    cb.ale('CX 19 4 20 3 21 12 22 2')
    cb.time_tick()
    cb.ale('CX 19 5 20 8 21 4 22 10')
    cb.time_tick()
    cb.ale('CX 19 6 20 9 21 5 22 12')
    cb.time_tick()
    cb.ale('CX 19 7 20 10 21 13 22 14')
    cb.ale('H 19 20 21 22')
    #cb.ale(f'MR({noise}) 19 20 21 22')
    cb.al('MR(0) 19 20 21 22')
    cb.ale('H 19 20 21 22')

    cb.begin_loop(nrounds)
    # Z
    cb.time_tick()
    cb.ale('CX 0 15 11 16 8 17 4 18')
    cb.time_tick()
    cb.ale('CX 1 15 0 16 9 17 6 18')
    cb.time_tick()
    cb.ale('CX 2 15 1 16 0 17 8 18')
    cb.time_tick()
    cb.ale('CX 3 15 2 16 1 17 0 18')
    cb.time_tick()
    cb.ale('CX 4 15 3 16 12 17 2 18')
    cb.time_tick()
    cb.ale('CX 5 15 8 16 4 17 10 18')
    cb.time_tick()
    cb.ale('CX 6 15 9 16 5 17 12 18')
    cb.time_tick()
    cb.ale('CX 7 15 10 16 13 17 14 18')
    #cb.ale(f'MR({noise}) 15 16 17 18') 
    cb.al('MR(0) 15 16 17 18')
    cb.add_detectors([[-1, -9], [-2, -10], [-3, -11], [-4, -12]])
    # Z   
    cb.time_tick()
    cb.ale('CX 19 0 20 11 21 8 22 4')
    cb.time_tick()
    cb.ale('CX 19 1 20 0 21 9 22 6')
    cb.time_tick()
    cb.ale('CX 19 2 20 1 21 0 22 8')
    cb.time_tick()
    cb.ale('CX 19 3 20 2 21 1 22 0')
    cb.time_tick()
    cb.ale('CX 19 4 20 3 21 12 22 2')
    cb.time_tick()
    cb.ale('CX 19 5 20 8 21 4 22 10')
    cb.time_tick()
    cb.ale('CX 19 6 20 9 21 5 22 12')
    cb.time_tick()
    cb.ale('CX 19 7 20 10 21 13 22 14')
    cb.ale('H 19 20 21 22')
    #cb.ale(f'MR({noise}) 19 20 21 22')
    cb.al('MR(0) 19 20 21 22')
    cb.ale('H 19 20 21 22')
    cb.add_detectors([[-1, -9], [-2, -10], [-3, -11], [-4, -12]])
    
    cb.end_loop()

    # Final measurement of all physical qubits to give logical readout
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    cb.add_observable([-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15])

    return cb

def hamming15decomposition(noise, nrounds):
    cb = CircuitBuilder.CircuitBuilder(noise, noise, noise, noise, noise, noise)

    # State preparation
    cb.al('R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    #cb.al('X 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14') # For preparing all 1s
    cb.al('R 15 16 17 18 19 20 21 22 23 24 25 26')
    cb.al('H 21 22 23 24 25 26') # Rotate X type ancilla to X basis

    # 1234
    #     1234
    #         1234
    # 2   3   4   1
    #  3   4   1   2
    #   4   1   2   3

    cb.time_tick()
    cb.ale('CX 0 15 4 16 8 17 12 18 9 19 6 20')
    cb.time_tick()
    cb.ale('CX 1 15 5 16 9 17 0 18 13 19 10 20')
    cb.time_tick()
    cb.ale('CX 2 15 6 16 10 17 4 18 1 19 14 20')
    cb.time_tick()
    cb.ale('CX 3 15 7 16 11 17 8 18 5 19 2 20')

    #cb.ale(f'MR({noise}) 15 16 17 18 19 20')
    cb.al('MR(0) 15 16 17 18 19 20')

    cb.time_tick()
    cb.ale('CX 21 0 22 4 23 8 24 12 25 9 26 6')
    cb.time_tick()
    cb.ale('CX 21 1 22 5 23 9 24 0 25 13 26 10')
    cb.time_tick()
    cb.ale('CX 21 2 22 6 23 10 24 4 25 1 26 14')
    cb.time_tick()
    cb.ale('CX 21 3 22 7 23 11 24 8 25 5 26 2')
    cb.ale('H 21 22 23 24 25 26')
    #cb.ale(f'MR({noise}) 21 22 23 24 25 26')
    cb.al('MR(0) 21 22 23 24 25 26')
    cb.ale('H 21 22 23 24 25 26')

    cb.begin_loop(nrounds)
    
    cb.time_tick()
    cb.ale('CX 0 15 4 16 8 17 12 18 9 19 6 20')
    cb.time_tick()
    cb.ale('CX 1 15 5 16 9 17 0 18 13 19 10 20')
    cb.time_tick()
    cb.ale('CX 2 15 6 16 10 17 4 18 1 19 14 20')
    cb.time_tick()
    cb.ale('CX 3 15 7 16 11 17 8 18 5 19 2 20')
    #cb.ale(f'MR({noise}) 15 16 17 18 19 20')
    cb.al('MR(0) 15 16 17 18 19 20')

    cb.time_tick()
    cb.ale('CX 21 0 22 4 23 8 24 12 25 9 26 6')
    cb.time_tick()
    cb.ale('CX 21 1 22 5 23 9 24 0 25 13 26 10')
    cb.time_tick()
    cb.ale('CX 21 2 22 6 23 10 24 4 25 1 26 14')
    cb.time_tick()
    cb.ale('CX 21 3 22 7 23 11 24 8 25 5 26 2')
    cb.ale('H 21 22 23 24 25 26')
    #cb.ale(f'MR({noise}) 21 22 23 24 25 26')
    cb.al('MR(0) 21 22 23 24 25 26')
    cb.ale('H 21 22 23 24 25 26')

    cb.end_loop()

    # Final measurement of all physical qubits to give logical readout
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    cb.add_observable([-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15])

    return cb
    

In [8]:
# Util functions
def get_logical_error(circuit, nshots=1000000, batch_size=100000):
    ds = circuit.compile_detector_sampler()
    total_errors = 0
    total_shots = 0
    model = circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(model)

    shots_remaining = nshots
    while shots_remaining > 0:
        current_batch_size = int(min(batch_size, shots_remaining))
        D, L = ds.sample(current_batch_size, separate_observables=True)
        pred = matching.decode_batch(D)
        total_errors += np.sum(pred[:, 0] != L[:, 0])
        total_shots += current_batch_size
        shots_remaining -= current_batch_size

    return total_errors / total_shots

# xor two bitstrings
def xor(a, b):
    if len(a) != len(b):
        raise ValueError("Bitstrings must be of the same length")
    return ''.join(str(int(a[i]) ^ int(b[i])) for i in range(len(a)))

def todecimal(a):
    return int(''.join(str(int(bit)) for bit in a), 2)

def tobitstring(n):
    return format(n, 'b')

def bitflip(bitstring, idx):
    l = list(bitstring)
    #print(l)
    l[idx] = '1' if l[idx] == '0' else '0'
    return ''.join(l)

def sum_bitstring(bitstring):
    return sum([1 for x in bitstring if int(x) == 1])

def count_differences(bitstring1, bitstring2):
    return sum(1 for bit1, bit2 in zip(bitstring1, bitstring2) if bit1 != bit2)


In [4]:
nrounds = 1
er = 0.001
c = hamming15(er, nrounds).get_circuit()
sample = c.compile_detector_sampler().sample(1000000)
print(sample[0:5])
print('Total errors: ' + str((sample.sum(axis=1) >= 1).sum()))
print('Single errors: ' + str((sample.sum(axis=1) == 1).sum()))
print('Logical error rate: ' + str(get_logical_error(c)))

[[False False False False  True False False False]
 [False False False False False False False False]
 [False False False False False False False False]
 [False False  True  True False False False False]
 [False False False False False False False False]]
Total errors: 441938
Single errors: 179871
Logical error rate: 0.237542


In [5]:
# Workcell for hamming15 manual decoding
nrounds = 1
er = 0.001
c = hamming15(er, nrounds)
single = c.sample(1)[0][0]
sr0, sr1 = single[0:8], single[8:16]
det = xor(sr0, sr1)
print(single)
print(sr0, sr1)
print(det, det[0:4], todecimal(det[0:4]), det[4:8], todecimal(det[4:8]))
lr = single[16:]
print(f'{lr[0:5]} {lr[5:10]} {lr[10:15]}')
if todecimal(det[0:4]) != 0:
    lr = bitflip(lr, 15 - todecimal(det[0:4]))
print(f'{lr[0:5]} {lr[5:10]} {lr[10:15]}')
most_similar = most_similar_codeword(lr)
print(f'{most_similar[0:5]} {most_similar[5:10]} {most_similar[10:15]}')
lrz = sum([1 for x in lr if int(x) == 1]) % 2
print(lrz)



0000011100000111011001100110011
00000111 00000111
00000000 0000 0 0000 0
01100 11001 10011
01100 11001 10011


NameError: name 'most_similar_codeword' is not defined

In [6]:
# Workcell for hamming15decomposition
nrounds, er = 1, 0.001
c = hamming15decomposition(er, nrounds)
sample = c.sample(1)[0][0]
g0, g1 = sample[0:12], sample[12:24]
print(g0, g1)
g0z0, g0z1, g0z2, g0z3 = xor(g0[0], g0[1]), xor(g0[0], g0[2]), xor(g0[3], g0[4]), xor(g0[3], g0[5])
g0x0, g0x1, g0x2, g0x3 = xor(g0[6], g0[7]), xor(g0[6], g0[8]), xor(g0[9], g0[10]), xor(g0[9], g0[11])
g1z0, g1z1, g1z2, g1z3 = xor(g1[0], g1[1]), xor(g1[0], g1[2]), xor(g1[3], g1[4]), xor(g1[3], g1[5])
g1x0, g1x1, g1x2, g1x3 = xor(g1[6], g1[7]), xor(g1[6], g1[8]), xor(g1[9], g1[10]), xor(g1[9], g1[11])
# Reconstructed stabilizer measurements
s0 = ''.join([g0z0, g0z1, g0z2, g0z3, g0x0, g0x1, g0x2, g0x3])
s1 = ''.join([g1z0, g1z1, g1z2, g1z3, g1x0, g1x1, g1x2, g1x3])
print(s0, s1)
det = xor(s0, s1)
print(det, det[0:4], todecimal(det[0:4]), det[4:8], todecimal(det[4:8]))
lr = sample[24:]
print(f'{lr[0:5]} {lr[5:10]} {lr[10:15]}')
if todecimal(det[0:4]) != 0:
    lr = bitflip(lr, 15 - todecimal(det[0:4]))
print(f'{lr[0:5]} {lr[5:10]} {lr[10:15]}')
most_similar = most_similar_codeword_decomp(lr)
print(f'{most_similar[0:5]} {most_similar[5:10]} {most_similar[10:15]}')
lrz = sum([1 for x in lr if int(x) == 1]) % 2
print(lrz)


000000000010 111000000101
00000010 00000010
00000000 0000 0 0000 0
01110 11110 00100
01110 11110 00100


NameError: name 'most_similar_codeword_decomp' is not defined

In [9]:
# Codewords basis hamming15
nrounds, er = 1, 0
c = hamming15(er, nrounds)
sample = c.sample(1000000)
codewords = [sample[i][0][16:] for i in range(16)]

def most_similar_codeword(codeword):
    least_differences = 15
    for i in range(16):
        xored = xor(codeword, codewords[i])
        differences = sum(1 for bit in xored if bit == '1')
        if differences < least_differences:
            least_differences = differences
            most_similar_codeword = codewords[i]
    return most_similar_codeword

def hamming15experimentZ(nrounds = 1, er = 0.001, nsamples = 100000, error_correction = True):
    c = hamming15(er, nrounds)
    # Manual decoding
    logical_errors = 0
    outside_codespace = 0
    detector_fires = 0
    detector_fires_Z = 0
    detector_fires_X = 0
    successful_corrections = 0
    harmful_corrections = 0
    for _ in range(nsamples):
        single = c.sample(1)[0][0]
        sprev = single[0:8]
        lr = single[8+8*nrounds:]
        detector_has_fired_Z = False
        detector_has_fired_X = False
        detector_has_fired = False
        for n in range(nrounds):
            scurr = single[8*n + 8:8*(n+1) + 8]
            det = xor(sprev, scurr)
            if error_correction:
                if todecimal(det[0:4]) != 0:
                    newlr = bitflip(lr, 15 - todecimal(det[0:4]))
                    if count_differences(newlr, most_similar_codeword(newlr)) == 0:
                        successful_corrections += 1
                    if count_differences(lr, most_similar_codeword(lr)) == 0 and count_differences(newlr, most_similar_codeword(newlr)) > 0:
                        harmful_corrections += 1
            sprev = scurr
            if sum_bitstring(det[0:8]) > 0 and not detector_has_fired:
                detector_has_fired = True
                detector_fires += 1
            if sum_bitstring(det[0:4]) > 0 and not detector_has_fired_Z:
                detector_has_fired_Z = True
                detector_fires_Z += 1
            if sum_bitstring(det[4:8]) > 0 and not detector_has_fired_X:
                detector_has_fired_X = True
                detector_fires_X += 1
        lrz = sum_bitstring(lr) % 2
        if lrz == 1:
            logical_errors += 1
        if count_differences(lr, most_similar_codeword(lr)) > 0:
            outside_codespace += 1

    return logical_errors/nsamples, outside_codespace/nsamples, detector_fires/nsamples, detector_fires_Z/nsamples, detector_fires_X/nsamples, successful_corrections/nsamples, harmful_corrections/nsamples

In [10]:
# Codewords basis hamming15decomposition
nrounds, er = 1, 0
c = hamming15decomposition(er, nrounds)
sample = c.sample(10000000)
codewordsdecomp = [sample[i][0][24:] for i in range(64)]

def most_similar_codeword_decomp(codeword):
    least_differences = 15
    for i in range(64):
        xored = xor(codeword, codewordsdecomp[i])
        differences = sum(1 for bit in xored if bit == '1')
        if differences < least_differences:
            least_differences = differences
            most_similar_codeword = codewordsdecomp[i]
    return most_similar_codeword

def hamming15decompositionexperimentZ(nrounds = 1, er = 0.001, nsamples = 100000, error_correction = True):
    c = hamming15decomposition(er, nrounds)
    # Manual decoding
    logical_errors = 0
    outside_codespace = 0
    detector_fires = 0
    detector_fires_Z = 0
    detector_fires_X = 0
    successful_corrections = 0
    harmful_corrections = 0
    for _ in range(nsamples):
        single = c.sample(1)[0][0]
        sprev = single[0:12]
        lr = single[12+12*nrounds:]
        detector_has_fired_Z = False
        detector_has_fired_X = False
        detector_has_fired = False
        for n in range(nrounds):
            scurr = single[12*n + 12:12*(n+1) + 12]
            g0, g1 = sprev, scurr
            g0z0, g0z1, g0z2, g0z3 = xor(g0[0], g0[1]), xor(g0[0], g0[2]), xor(g0[3], g0[4]), xor(g0[3], g0[5])
            g0x0, g0x1, g0x2, g0x3 = xor(g0[6], g0[7]), xor(g0[6], g0[8]), xor(g0[9], g0[10]), xor(g0[9], g0[11])
            g1z0, g1z1, g1z2, g1z3 = xor(g1[0], g1[1]), xor(g1[0], g1[2]), xor(g1[3], g1[4]), xor(g1[3], g1[5])
            g1x0, g1x1, g1x2, g1x3 = xor(g1[6], g1[7]), xor(g1[6], g1[8]), xor(g1[9], g1[10]), xor(g1[9], g1[11])
            # Reconstructed stabilizer measurements
            s0 = ''.join([g0z0, g0z1, g0z2, g0z3, g0x0, g0x1, g0x2, g0x3])
            s1 = ''.join([g1z0, g1z1, g1z2, g1z3, g1x0, g1x1, g1x2, g1x3])
            det = xor(s0, s1)
            if error_correction:
                if todecimal(det[0:4]) != 0:
                    newlr = bitflip(lr, 15 - todecimal(det[0:4]))
                    if count_differences(newlr, most_similar_codeword_decomp(newlr)) == 0:
                        successful_corrections += 1
                    if count_differences(lr, most_similar_codeword_decomp(lr)) == 0 and count_differences(newlr, most_similar_codeword_decomp(newlr)) > 0:
                        harmful_corrections += 1
            sprev = scurr
            if sum_bitstring(det[0:8]) > 0 and not detector_has_fired:
                detector_has_fired = True
                detector_fires += 1
            if sum_bitstring(det[0:4]) > 0 and not detector_has_fired_Z:
                detector_has_fired_Z = True
                detector_fires_Z += 1
            if sum_bitstring(det[4:8]) > 0 and not detector_has_fired_X:
                detector_has_fired_X = True
                detector_fires_X += 1
        lrz = sum_bitstring(lr) % 2
        if lrz == 1:
            logical_errors += 1
        if count_differences(lr, most_similar_codeword_decomp(lr)) > 0:
            outside_codespace += 1

    return logical_errors/nsamples, outside_codespace/nsamples, detector_fires/nsamples, detector_fires_Z/nsamples, detector_fires_X/nsamples, successful_corrections/nsamples, harmful_corrections/nsamples

In [11]:
nrounds = [1, 2, 3, 4, 5, 10, 20, 30, 40, 50]
ers = [0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005]
results_with_correction = [[[] for _ in nrounds] for _ in ers]
results_without_correction = [[[] for _ in nrounds] for _ in ers]
for i, er in enumerate(ers):
    for j, nround in tqdm(enumerate(nrounds)):
        results_with_correction[i][j] = hamming15experimentZ(nround, er, 10000, True)
        results_without_correction[i][j] = hamming15experimentZ(nround, er, 10000, False)
with open('hamming15_results_with_correction.pkl', 'wb') as f:
    pickle.dump(results_with_correction, f)
with open('hamming15_results_without_correction.pkl', 'wb') as f:
    pickle.dump(results_without_correction, f)


0it [00:00, ?it/s]

10it [00:49,  4.90s/it]
10it [00:57,  5.72s/it]
10it [01:16,  7.60s/it]
10it [01:43, 10.38s/it]
10it [02:26, 14.60s/it]
10it [03:27, 20.73s/it]


In [12]:
nrounds = [1, 2, 3, 4, 5, 10, 20, 30, 40, 50]
ers = [0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005]
results_with_correction_decomp = [[[] for _ in nrounds] for _ in ers]
results_without_correction_decomp = [[[] for _ in nrounds] for _ in ers]
for i, er in enumerate(ers):
    for j, nround in tqdm(enumerate(nrounds)):
        results_with_correction_decomp[i][j] = hamming15decompositionexperimentZ(nround, er, 10000, True)
        results_without_correction_decomp[i][j] = hamming15decompositionexperimentZ(nround, er, 10000, False)
with open('hamming15_results_with_correction_decomp.pkl', 'wb') as f:
    pickle.dump(results_with_correction_decomp, f)
with open('hamming15_results_without_correction_decomp.pkl', 'wb') as f:
    pickle.dump(results_without_correction_decomp, f)

   

10it [02:25, 14.53s/it]
10it [02:42, 16.25s/it]
10it [03:31, 21.16s/it]
10it [04:44, 28.49s/it]
10it [06:49, 40.98s/it]
10it [10:47, 64.77s/it] 


In [ ]:

# Save results to file with pickle
with open('hamming15_results_with_correction.pkl', 'wb') as f:
    pickle.dump(results_with_correction, f)

with open('hamming15_results_without_correction.pkl', 'wb') as f:
    pickle.dump(results_without_correction, f)

In [674]:
results_with_correction

[[(0.0365, 0.0401, 0.0566, 0.0326, 0.035, 0.0146, 0.0088),
  (0.0561, 0.0612, 0.0961, 0.0581, 0.0623, 0.03, 0.0183),
  (0.0784, 0.0883, 0.1317, 0.0864, 0.0849, 0.047, 0.0257),
  (0.0907, 0.1011, 0.1633, 0.1058, 0.1071, 0.0613, 0.0348),
  (0.1061, 0.1212, 0.2005, 0.1295, 0.1341, 0.0698, 0.0409),
  (0.1844, 0.2091, 0.344, 0.2356, 0.2419, 0.1304, 0.0815),
  (0.2974, 0.3806, 0.5774, 0.4289, 0.4307, 0.2203, 0.1261),
  (0.3629, 0.4886, 0.7136, 0.5548, 0.5608, 0.2566, 0.1634),
  (0.404, 0.584, 0.8136, 0.6648, 0.6696, 0.2753, 0.2038),
  (0.4344, 0.6607, 0.8771, 0.7418, 0.7516, 0.2795, 0.2035)],
 [(0.073, 0.0823, 0.111, 0.0618, 0.0706, 0.0261, 0.0149),
  (0.1111, 0.126, 0.1841, 0.1127, 0.1188, 0.0558, 0.0309),
  (0.1373, 0.1574, 0.2383, 0.1556, 0.1558, 0.0794, 0.0509),
  (0.1716, 0.199, 0.3082, 0.2045, 0.2101, 0.1053, 0.0601),
  (0.1908, 0.2273, 0.3521, 0.239, 0.248, 0.1221, 0.0762),
  (0.2905, 0.3677, 0.5732, 0.4139, 0.4308, 0.1991, 0.1393),
  (0.4153, 0.594, 0.8118, 0.6614, 0.667, 0.2763, 0.1

In [11]:
def hamming15injectError(mode, noise):
    error_qubits_z = []
    error_qubits_x = []
    if mode == 0: # Inject error for only physical qubits
        for i in range(15):
            if random.random() < noise:
                error_qubits_z.append(i)
            if random.random() < noise:
                error_qubits_x.append(i)
    elif mode == 1: # Inject error for only ancilla qubits
        for i in range(8):
            if random.random() < noise:
                error_qubits_z.append(i + 15)
            if random.random() < noise:
                error_qubits_x.append(i + 15)
    elif mode == 2: # Inject error for both physical and ancilla qubits
        for i in range(23):
            if random.random() < noise:
                error_qubits_z.append(i)
            if random.random() < noise:
                error_qubits_x.append(i)

    cb = CircuitBuilder.CircuitBuilder(0, 0, 0, 0, 0, 0)
    cb.al('R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    cb.al('R 15 16 17 18 19 20 21 22')
    cb.al('H 19 20 21 22')
    
    cb.ale('CX 0 15 11 16 8 17 4 18')
    cb.ale('CX 1 15 0 16 9 17 6 18')
    cb.ale('CX 2 15 1 16 0 17 8 18')
    cb.ale('CX 3 15 2 16 1 17 0 18')
    cb.ale('CX 4 15 3 16 12 17 2 18')
    cb.ale('CX 5 15 8 16 4 17 10 18')
    cb.ale('CX 6 15 9 16 5 17 12 18')
    cb.ale('CX 7 15 10 16 13 17 14 18')
    cb.ale('MR(0) 15 16 17 18')    
    cb.ale('CX 19 0 20 11 21 8 22 4')
    cb.ale('CX 19 1 20 0 21 9 22 6')
    cb.ale('CX 19 2 20 1 21 0 22 8')
    cb.ale('CX 19 3 20 2 21 1 22 0')
    cb.ale('CX 19 4 20 3 21 12 22 2')
    cb.ale('CX 19 5 20 8 21 4 22 10')
    cb.ale('CX 19 6 20 9 21 5 22 12')
    cb.ale('CX 19 7 20 10 21 13 22 14')
    cb.ale('H 19 20 21 22')
    cb.ale('MR(0) 19 20 21 22')
    cb.ale('H 19 20 21 22')

    # Inject error between warmup and first round
    if len(error_qubits_x) > 0:
        cb.al('X ' + ' '.join([str(i) for i in error_qubits_x]))
    if len(error_qubits_z) > 0:
        cb.al('Z ' + ' '.join([str(i) for i in error_qubits_z]))
    cb.ale('CX 0 15 11 16 8 17 4 18')
    cb.ale('CX 1 15 0 16 9 17 6 18')
    cb.ale('CX 2 15 1 16 0 17 8 18')
    cb.ale('CX 3 15 2 16 1 17 0 18')
    cb.ale('CX 4 15 3 16 12 17 2 18')
    cb.ale('CX 5 15 8 16 4 17 10 18')
    cb.ale('CX 6 15 9 16 5 17 12 18')
    cb.ale('CX 7 15 10 16 13 17 14 18')
    cb.ale('MR(0) 15 16 17 18')   
    cb.add_detectors([[-1, -9], [-2, -10], [-3, -11], [-4, -12]])
    cb.ale('CX 19 0 20 11 21 8 22 4')
    cb.ale('CX 19 1 20 0 21 9 22 6')
    cb.ale('CX 19 2 20 1 21 0 22 8')
    cb.ale('CX 19 3 20 2 21 1 22 0')
    cb.ale('CX 19 4 20 3 21 12 22 2')
    cb.ale('CX 19 5 20 8 21 4 22 10')
    cb.ale('CX 19 6 20 9 21 5 22 12')
    cb.ale('CX 19 7 20 10 21 13 22 14')
    cb.ale('H 19 20 21 22')
    cb.ale('MR(0) 19 20 21 22')
    cb.ale('H 19 20 21 22')
    cb.add_detectors([[-1, -9], [-2, -10], [-3, -11], [-4, -12]])

    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    cb.add_observable([-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15])

    return cb, error_qubits_x, error_qubits_z

def hamming15decompositionInjectError(mode, noise):
    error_qubits_z = []
    error_qubits_x = []
    if mode == 0: # Inject error for only physical qubits
        for i in range(15):
            if random.random() < noise:
                error_qubits_z.append(i)
            if random.random() < noise:
                error_qubits_x.append(i)
    elif mode == 1: # Inject error for only ancilla qubits
        for i in range(12):
            if random.random() < noise:
                error_qubits_z.append(i + 15)
            if random.random() < noise:
                error_qubits_x.append(i + 15)
    elif mode == 2: # Inject error for both physical and ancilla qubits
        for i in range(27):
            if random.random() < noise:
                error_qubits_z.append(i)
            if random.random() < noise:
                error_qubits_x.append(i)
    

    cb = CircuitBuilder.CircuitBuilder(0, 0, 0, 0, 0, 0)
    cb.al('R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    #cb.al('X 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14') # For preparing all 1s
    cb.al('R 15 16 17 18 19 20 21 22 23 24 25 26')
    cb.al('H 21 22 23 24 25 26') # Rotate X type ancilla to X basis

    cb.ale('CX 0 15 4 16 8 17 12 18 9 19 6 20')
    cb.ale('CX 1 15 5 16 9 17 0 18 13 19 10 20')
    cb.ale('CX 2 15 6 16 10 17 4 18 1 19 14 20')
    cb.ale('CX 3 15 7 16 11 17 8 18 5 19 2 20')
    cb.al('MR(0) 15 16 17 18 19 20')
    cb.ale('CX 21 0 22 4 23 8 24 12 25 9 26 6')
    cb.ale('CX 21 1 22 5 23 9 24 0 25 13 26 10')
    cb.ale('CX 21 2 22 6 23 10 24 4 25 1 26 14')
    cb.ale('CX 21 3 22 7 23 11 24 8 25 5 26 2')
    cb.ale('H 21 22 23 24 25 26')
    cb.al('MR(0) 21 22 23 24 25 26')
    cb.ale('H 21 22 23 24 25 26')
    # Inject error between warmup and first round
    if len(error_qubits_x) > 0:
        cb.al('X ' + ' '.join([str(i) for i in error_qubits_x]))
    if len(error_qubits_z) > 0:
        cb.al('Z ' + ' '.join([str(i) for i in error_qubits_z]))
    cb.ale('CX 0 15 4 16 8 17 12 18 9 19 6 20')
    cb.ale('CX 1 15 5 16 9 17 0 18 13 19 10 20')
    cb.ale('CX 2 15 6 16 10 17 4 18 1 19 14 20')
    cb.ale('CX 3 15 7 16 11 17 8 18 5 19 2 20')
    cb.al('MR(0) 15 16 17 18 19 20')

    cb.ale('CX 21 0 22 4 23 8 24 12 25 9 26 6')
    cb.ale('CX 21 1 22 5 23 9 24 0 25 13 26 10')
    cb.ale('CX 21 2 22 6 23 10 24 4 25 1 26 14')
    cb.ale('CX 21 3 22 7 23 11 24 8 25 5 26 2')
    cb.ale('H 21 22 23 24 25 26')
    cb.al('MR(0) 21 22 23 24 25 26')
    cb.ale('H 21 22 23 24 25 26')

    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14')
    cb.add_observable([-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15])

    return cb, error_qubits_x, error_qubits_z


In [ ]:
# Workcell for hamming15
nrounds = 1
er = 0.05
c, error_qubits_x, error_qubits_z = hamming15injectError(2, er)
single = c.sample(1)[0][0]
sr0, sr1 = single[0:8], single[8:16]
det = xor(sr0, sr1)
print(single)
print('Physical', [15 - x for x in error_qubits_x if x < 15], [15 - x for x in error_qubits_z if x < 15])
print('Ancilla', [23 - x for x in error_qubits_x if x >= 15], [23 - x for x in error_qubits_z if x >= 15])
print(sr0, sr1)
print(det, det[0:4], todecimal(det[0:4]), det[4:8], todecimal(det[4:8]))
lr = single[16:]

print(f'{lr[0:5]} {lr[5:10]} {lr[10:15]}')
most_similar = most_similar_codeword(lr)
print(f'{most_similar[0:5]} {most_similar[5:10]} {most_similar[10:15]}')
lrz = sum([1 for x in lr if int(x) == 1]) % 2
print(lrz)

0000001111110011000110010110011
Physical [15] []
Ancilla [] []
00000011 11110011
11110000 1111 15 0000 0
00011 00101 10011
10011 00101 10011
1


In [15]:
# Workcell for hamming15decomposition
er = 0.0
c, error_qubits_x, error_qubits_z = hamming15decompositionInjectError(0, er)
single = c.sample(1)[0][0]
g0, g1 = single[0:12], single[12:24]
print(single)
print('Physical', [15 - x for x in error_qubits_x if x < 15], [15 - x for x in error_qubits_z if x < 15])
print('Ancilla', [27 - x for x in error_qubits_x if x >= 15], [27 - x for x in error_qubits_z if x >= 15])
print(g0, g1)
g0z0, g0z1, g0z2, g0z3 = xor(g0[0], g0[1]), xor(g0[0], g0[2]), xor(g0[3], g0[4]), xor(g0[3], g0[5])
g0x0, g0x1, g0x2, g0x3 = xor(g0[6], g0[7]), xor(g0[6], g0[8]), xor(g0[9], g0[10]), xor(g0[9], g0[11])
g1z0, g1z1, g1z2, g1z3 = xor(g1[0], g1[1]), xor(g1[0], g1[2]), xor(g1[3], g1[4]), xor(g1[3], g1[5])
g1x0, g1x1, g1x2, g1x3 = xor(g1[6], g1[7]), xor(g1[6], g1[8]), xor(g1[9], g1[10]), xor(g1[9], g1[11])
# Reconstructed stabilizer measurements
s0 = ''.join([g0z0, g0z1, g0z2, g0z3, g0x0, g0x1, g0x2, g0x3])
s1 = ''.join([g1z0, g1z1, g1z2, g1z3, g1x0, g1x1, g1x2, g1x3])
det = xor(s0, s1)
print('rec: ' +det )
print(det[0:4], todecimal(det[0:4]), det[4:8], todecimal(det[4:8]))
lr = single[24:]

print(f'{lr[0:5]} {lr[5:10]} {lr[10:15]}')
most_similar = most_similar_codeword_decomp(lr)
print(f'{most_similar[0:5]} {most_similar[5:10]} {most_similar[10:15]}')
lrz = sum([1 for x in lr if int(x) == 1]) % 2
print(lrz)

000000011011000000100100101101000100010
Physical [] []
Ancilla [] []
000000011011 000000100100
rec: 00000000
0000 0 0000 0
10110 10001 00010
10110 10001 00010
0


In [307]:
xor(tobitstring(15), tobitstring(14))

'0001'